In [43]:
import pandas as pd
import numpy as np
data = pd.read_csv("pakistan_economic_indicators_2000_2025.csv")


data.isnull().sum().sum()
data.head(5)



,year,gdp_usd_bn,gdp_growth_pct,gdp_per_capita_usd,inflation_cpi_pct,unemployment_pct,remittances_usd_bn,exports_usd_bn,imports_usd_bn,trade_balance_usd_bn,...,mobile_per_100,imf_program_active,key_events,remittances_gdp_pct,exports_gdp_pct,imports_gdp_pct,fdi_gdp_pct,gdp_growth_category,inflation_category,decade
0,2000,99.90,3.90,624,4.40,7.80,1.08,9.20,10.30,-1.10,...,0.30,0,Post-sanctions recovery,1.08,9.21,10.31,0.31,Moderate Growth,Low,2000s
1,2001,101.40,2.00,614,3.10,7.80,1.46,9.80,10.30,-0.50,...,0.60,0,9/11 impact + debt relief,1.44,9.66,10.16,0.37,Low Growth,Low,2000s
2,2002,107.60,3.10,628,3.50,8.30,2.39,10.00,10.90,-0.90,...,1.80,0,Economic reform start,2.22,9.29,10.13,0.76,Moderate Growth,Low,2000s
3,2003,116.30,4.70,657,3.10,8.30,3.96,11.70,12.20,-0.50,...,3.30,0,Growth acceleration,3.40,10.06,10.49,0.46,Moderate Growth,Low,2000s
4,2004,126.70,7.40,697,7.40,7.70,3.87,14.00,15.50,-1.50,...,7.70,0,High growth period,3.05,11.05,12.23,0.88,High Growth,Moderate,2000s


In [8]:
data.shape


(26, 32)

In [59]:
df.groupby("decade")["gdp_growth_pct"].mean()


df.groupby(["decade", "imf_program_active"])["gdp_growth_pct"].mean()

ans = df.groupby(["decade", "imf_program_active"]).agg({
    "gdp_growth_pct": "mean",
    "inflation_cpi_pct": "mean",
    "fdi_inflows_usd_bn": "sum"    
})

print(ans)


                           gdp_growth_pct  inflation_cpi_pct  fdi_inflows_usd_bn
decade imf_program_active                                                       
2000s  0                             5.11               6.34               15.07
       1                             4.70              14.20                7.93
2010s  0                             3.48               6.97               10.74
       1                             4.75               8.17                7.82
2020s  0                             6.10              12.20                1.93
       1                             2.11              15.52                9.95


## Question 1: What were the Stagflation years? 


In [40]:
stagflation = df.loc[
    (df["inflation_cpi_pct"] > 10) & (df["gdp_growth_pct"] < 3),
    ["year", "gdp_growth_pct", "inflation_cpi_pct", "policy_rate_pct", "imf_program_active", "key_events"]
].reset_index(drop=True)



stagflation

,year,gdp_growth_pct,inflation_cpi_pct,policy_rate_pct,imf_program_active,key_events
0,2009,2.60,20.80,12.50,1,IMF bailout + floods
1,2010,1.60,11.70,12.50,0,Floods recovery
2,2023,-0.04,29.20,22.00,1,IMF bailout + PKR crash
3,2024,2.40,23.40,13.00,1,Stabilization + disinflation


### Question 1 Analysis: Stagflation Years

**2009:** The global financial crisis hit Pakistan while it was already fragile. Growth collapsed to 2.6% while inflation ran at 20.8%. The SBP held rates at 12.5% which is too high to stimulate growth, not high enough to kill inflation. IMF bailout was unavoidable.

**2010:** Floods destroyed crops and infrastructure. Growth slowed further to 1.6%, inflation stayed elevated at 11.7%. No IMF programme active this year

**2023:** The worst year in the dataset. Growth went negative at -0.04% while inflation hit 29.2% which was the highest in 25 years. Policy rate raised to 22%, the highest ever, meaning borrowing became nearly impossible for businesses. PKR crashed to 256. Reserves at \$8.7bn. Everything went wrong simultaneously.

**2024:** Still in crisis but stabilising. Inflation at 23.4% is still devastating but falling from 2023's peak. Growth returned slightly at 2.4%. The disinflation process had begun but ordinary Pakistanis were still facing prices roughly double what they were in 2021.



 ## Question 2: REMITTANCES vs CURRENT ACCOUNT: Do inflows actually close the gap?

In [30]:
rem = df[["year", "remittances_usd_bn", "current_account_usd_bn"]].copy()


rem["coverage_ratio"] = rem.apply(
    lambda row: row["remittances_usd_bn"] / abs(row["current_account_usd_bn"])
    if row["current_account_usd_bn"] < 0 else None,
    axis=1
)

rem["ca_status"] = rem["current_account_usd_bn"].apply(
    lambda x: "surplus" if x > 0 else "deficit"
)

rem

,year,remittances_usd_bn,current_account_usd_bn,coverage_ratio,ca_status
0,2000,1.08,-0.20,5.40,deficit
1,2001,1.46,0.30,NaN,surplus
2,2002,2.39,3.90,NaN,surplus
3,2003,3.96,4.10,NaN,surplus
4,2004,3.87,1.50,NaN,surplus
5,2005,4.28,-1.50,2.85,deficit
6,2006,5.12,-5.00,1.02,deficit
7,2007,6.14,-6.90,0.89,deficit
8,2008,6.45,-13.90,0.46,deficit
9,2009,7.81,-3.90,2.00,deficit


### Question 2 Analysis

**2000–2008:** Remittances were too small to matter much. Coverage ratios of 0.46–2.85, but the current account was also small. The economy wasn't running large deficits yet, so this wasn't a crisis.

**2007–2008:** The ratio drops below 1. The current account deficit ballooned to -\$6.9bn and -\$13.9bn as import demand surged. Remittances were growing but nowhere near fast enough. This is what pushed Pakistan to the IMF in 2009.

**2011 (the outlier):** Ratio of 56, not because remittances were enormous, but because the current account was nearly balanced (+$0.20bn). Remittances of \$11bn against almost no deficit produces an astronomical ratio. Don't read too much into it.

**2017–2018:** The ratio collapses to 1.56 and then 1.06 — the most important signal in the whole table. 
Remittances hit \$19–21bn but the current account deficit exploded to -\$12.6bn and -\$19.9bn. CPEC imports, energy imports, and consumption all surged simultaneously. The lifeline nearly snapped. This is what caused the 2019 IMF programme.

**2020:** Ratio jumps to 12.17. COVID crushed imports so the deficit shrank to -\$1.9bn while remittances held steady. Looks great but it's a lockdown artifact, not a structural improvement.

**2024:** Ratio of 49.43. An outlier similar to 2011. Current account swung to a small surplus +\$0.70bn due to IMF-mandated import compression. Remittances of \$34.6bn against a near-zero deficit gives a misleadingly high ratio.

In [17]:
rem["financing_gap"] = rem["current_account_usd_bn"] + rem["remittances_usd_bn"]
rem

,year,remittances_usd_bn,current_account_usd_bn,coverage_ratio,ca_status,financing_gap
0,2000,1.08,-0.20,5.40,deficit,0.88
1,2001,1.46,0.30,NaN,surplus,1.76
2,2002,2.39,3.90,NaN,surplus,6.29
3,2003,3.96,4.10,NaN,surplus,8.06
4,2004,3.87,1.50,NaN,surplus,5.37
5,2005,4.28,-1.50,2.85,deficit,2.78
6,2006,5.12,-5.00,1.02,deficit,0.12
7,2007,6.14,-6.90,0.89,deficit,-0.76
8,2008,6.45,-13.90,0.46,deficit,-7.45
9,2009,7.81,-3.90,2.00,deficit,3.91


The financing_gap tells you the actual pressure on reserves and external borrowing far more honestly than the coverage ratio ever could.

## Question 3: Forex Reserve Danger years.. When did the buffer run dangerously low?

In [18]:
df["import_cover_months"] = df["forex_reserves_usd_bn"] / (df["imports_usd_bn"] / 12)

low_fx = df.loc[
    df["forex_reserves_usd_bn"] < 10,
    ["year", "forex_reserves_usd_bn", "import_cover_months", "pkr_per_usd", "imf_program_active", "key_events"]
].reset_index(drop=True)

low_fx

,year,forex_reserves_usd_bn,import_cover_months,pkr_per_usd,imf_program_active,key_events
0,2000,1.70,1.98,51.80,0,Post-sanctions recovery
1,2001,3.00,3.50,61.40,0,9/11 impact + debt relief
2,2002,4.80,5.28,59.70,0,Economic reform start
3,2003,9.50,9.34,57.80,0,Growth acceleration
4,2023,8.70,1.89,256.00,1,IMF bailout + PKR crash


### Question 3 Analaysis 

**2000:** Broke from sanctions. Under 2 months cover. Survived on fumes.

**2001:** 9/11 changed everything. US needed Pakistan, sanctions lifted, debt rescheduled. Reserves nearly doubled overnight. Nothing to do with economic management.

**2002:**  Aid kept flowing. Reserves hit $4.8bn, cover crossed 5 months. PKR actually strengthened in one if the only times in 25 years.


**2003:** 9.34 months cover. Crisis fully over. This was the peak of the post-9/11 dividend. Reserves grew faster than imports for three straight years.

**2023:** Back to crisis. 8.7bn sounds bigger than 2000's $1.7bn but cover is actually 1.89 months -which is worse . Economy is now 4× larger so it needs far more dollars to function. PKR at 256, down from 51 in 2000. IMF was the only option. No geopolitical windfall this time.

## Question 4: currency collpase: PKR depreciation and its economic fallout

In [21]:
df["pkr_chg_pct"] = df["pkr_per_usd"].pct_change() * 100

sharp = df.loc[
    df["pkr_chg_pct"] > 10,
    ["year", "pkr_per_usd", "pkr_chg_pct", "inflation_cpi_pct", "forex_reserves_usd_bn", "key_events"]
].reset_index(drop=True)

print(f"PKR: {df['pkr_per_usd'].iloc[0]:.0f} in 2000  →  {df['pkr_per_usd'].iloc[-1]:.0f} in 2025")
print(f"Correlation (depreciation vs inflation): {df['pkr_chg_pct'].corr(df['inflation_cpi_pct']):.2f}\n")
sharp

PKR: 52 in 2000  →  280 in 2025
Correlation (depreciation vs inflation): 0.45



,year,pkr_per_usd,pkr_chg_pct,inflation_cpi_pct,forex_reserves_usd_bn,key_events
0,2001,61.40,18.53,3.10,3.00,9/11 impact + debt relief
1,2009,78.50,25.40,20.80,13.00,IMF bailout + floods
2,2019,150.00,35.62,6.70,18.10,IMF stabilization
3,2022,204.90,25.78,12.20,13.70,Record inflation + floods
4,2023,256.00,24.94,29.20,8.70,IMF bailout + PKR crash


### Analysis Question 4 

Pakistan imports fuel, food, and machinery, so a weaker rupee immediately raises the cost of living. The 0.45 correlation confirms this link is real but not perfect (other factors like subsidies and price controls complicate it).
Five years where PKR fell more than 10% in a single year. 2001 was geopolitical shock, 2009 was BoP crisis, 2019 was IMF-forced correction, 2022–2023 was the worst sequence back to back. Cumulative loss since 2000: 441%. Each crash fed directly into inflation through import prices.

**Deeper Analysis of 2022 and 2023:**

2022: 25.8% drop. Floods destroyed crops, energy imports surged, political chaos paralysed decision making. Reserves fell from \$22bn  to  \$13.7bn in one year. Inflation hit 12.2% and was still accelerating.
2023: 24.9% drop on top of 2022's 25.8%. Two consecutive years of 25%+ depreciation is almost unheard of. By end of 2023 the rupee was at 256, reserves were at $8.7bn, and inflation hit 29.2% which is the worst in Pakistan's modern history. The IMF programme came with strict conditions including removing energy subsidies, which made inflation worse before it got better.

## Question 5: IMF dependency: Does Pakistan perform better inside or outside programmes?

In [22]:
imf_compare = (
    df.groupby("imf_program_active")[["gdp_growth_pct", "inflation_cpi_pct", "public_debt_gdp_pct", "tax_revenue_gdp_pct"]]
    .mean()
    .round(2)
    .rename(index={0: "No IMF programme", 1: "Under IMF programme"})
)

print(f"Years under IMF: {df['imf_program_active'].sum()}  |  Years without: {(df['imf_program_active']==0).sum()}\n")
imf_compare

Years under IMF: 11  |  Years without: 15



,gdp_growth_pct,inflation_cpi_pct,public_debt_gdp_pct,tax_revenue_gdp_pct
imf_program_active,,,,
No IMF programme,4.53,6.98,67.80,10.85
Under IMF programme,3.54,12.61,67.45,11.85


### Analysis Question 5:

Pakistan's stronger numbers outside IMF programmes simply reflect that programmes activate during crises, not cause them. The IMF gets called when reserves are gone and the rupee is collapsing - the damage is already done before conditions kick in. What the table genuinely exposes as a failure isn't the growth or inflation gap, it's the debt line. After 11 years of IMF programmes demanding fiscal discipline, debt-to-GDP sits at 67.45% - virtually identical to the 67.80% outside programmes. Pakistan takes the bailout, stabilises, loosens fiscal policy the moment growth returns, and ends up back at the same window. The IMF buys time. The structural problem - a tax base too narrow to service its own debt - remains unsolved.

## Question 6: How did the Sectoral GDP shift across decades?

In [61]:
sector = df.groupby("decade", observed=True)[
    ["agriculture_gdp_pct", "industry_gdp_pct", "services_gdp_pct", "gdp_growth_pct"]
].mean().round(2)

sector.columns = ["Agriculture %", "Industry %", "Services %", "Avg Growth %"]
sector

,Agriculture %,Industry %,Services %,Avg Growth %
decade,,,,
2000s,22.86,26.61,50.53,5.03
2010s,21.67,25.18,53.15,3.99
2020s,23.02,25.17,51.82,2.78


### Analysis Question 6:

Services have dominated throughout (50–53%), but notice agriculture isn't shrinking as fast as you'd expect from a "developing" economy. It's stuck around 22%. That's both a poverty story (a huge share of employment) and a vulnerability: when floods hit crops, the whole macro picture deteriorates fast.


## Question 7: FDI vs. GROWTH: When did Pakistan grow fast but still fail to attract investment?

In [28]:
fdi = df.loc[
    df["gdp_growth_pct"] >= 4,
    ["year", "gdp_growth_pct", "fdi_inflows_usd_bn", "fdi_gdp_pct", "key_events"]
].reset_index(drop=True)

below_1pct = (fdi["fdi_gdp_pct"] < 1.0).sum()
print(f"High-growth years (≥4%): {len(fdi)}  |  Of those, FDI below 1% of GDP: {below_1pct}\n")
fdi

High-growth years (≥4%): 14  |  Of those, FDI below 1% of GDP: 10



,year,gdp_growth_pct,fdi_inflows_usd_bn,fdi_gdp_pct,key_events
0,2003,4.70,0.53,0.46,Growth acceleration
1,2004,7.40,1.12,0.88,High growth period
2,2005,9.00,2.20,1.60,Record growth 8.6%
3,2006,5.80,4.27,2.80,Growth slowdown
4,2007,6.80,5.59,3.33,Global commodity boom
5,2008,5.00,5.44,2.83,Global financial crisis
6,2012,4.40,0.86,0.36,IMF program + energy crisis
7,2014,4.10,1.63,0.60,CPEC launch
8,2015,4.10,0.96,0.35,Commodity price fall
9,2016,5.50,2.58,0.91,Growth recovery


### Analysis Questinon 7

14 high-growth years, FDI below 1% of GDP in 10 of them. The 2005–2008 window was the only time Pakistan genuinely attracted serious foreign investment peaked at 5.6bn in 2007. Then the financial crisis hit, confidence collapsed, and FDI never recovered even as growth returned. CPEC promised to change this but the 2017 "investment peak" was still only $2.5bn and mostly debt-financed infrastructure, not private investors betting on Pakistan. Post-2012, Pakistan regularly grew at 4–6% while attracting less than 1% of GDP in FDI which means every growth episode was running on consumption, not investment.

**2003–2008** was the only real FDI story Pakistan has. Growth and FDI moved together. Musharraf-era liberalisation, telecom deregulation, and a stable security environment made Pakistan briefly attractive. $5.6bn in 2007 remains the all-time record. Notice FDI as % of GDP hit 3.33%, the only time it ever looked internationally competitive.

**2014–2017** CPEC facade: CPEC was announced as a 62bn investment game changer. The FDI numbers tell a different story - 1.63bn in 2014, 2.58bn peak in 2016, back down to $2.50bn in 2017. Most CPEC money came as loans to the Pakistani government, not as private FDI. The "investment peak" of the CPEC era was smaller than 2006, a full decade earlier.

## Question 8: debt trap: What is the relationship between public debt trajectory and tax revenue?

In [29]:
debt_tax = df[["year", "public_debt_gdp_pct", "tax_revenue_gdp_pct", "gdp_growth_pct", "imf_program_active"]].copy()
debt_tax["fiscal_gap"] = debt_tax["public_debt_gdp_pct"] - debt_tax["tax_revenue_gdp_pct"]

print("Decade averages:\n")
print(
    df.groupby("decade", observed=True)[["public_debt_gdp_pct", "tax_revenue_gdp_pct"]]
    .mean().round(2)
)
print("\nFull year-by-year view:")
debt_tax

Decade averages:

        public_debt_gdp_pct  tax_revenue_gdp_pct
decade                                          
2000s                 65.50                10.17
2010s                 66.00                11.55
2020s                 74.00                12.65

Full year-by-year view:


,year,public_debt_gdp_pct,tax_revenue_gdp_pct,gdp_growth_pct,imf_program_active,fiscal_gap
0,2000,82.00,9.30,3.90,0,72.70
1,2001,79.00,9.70,2.00,0,69.30
2,2002,75.00,10.10,3.10,0,64.90
3,2003,68.00,10.50,4.70,0,57.50
4,2004,65.00,10.50,7.40,0,54.50
5,2005,60.00,10.80,9.00,0,49.20
6,2006,56.00,10.90,5.80,0,45.10
7,2007,55.00,10.30,6.80,1,44.70
8,2008,58.00,10.50,5.00,0,47.50
9,2009,57.00,9.10,2.60,1,47.90


### Analysis Question 8

What fiscal gap means before anything else It's just debt minus tax revenue, both as % of GDP. A gap of 72.70 in 2000 means debt was 72.70 percentage points larger than what the government was collecting in tax. 

Debt started at 82% of GDP in 2000 and the government spent the entire 2000s bringing it down to 55% by 2007 which was the best stretch of fiscal management in 25 years. Then it climbed back up, peaked at 87% during COVID, and has only recently come back down to 65%. Tax collection improved from 9.3% to 14.5% over 25 years but painfully slowly and it took took a quarter century for that improvement. The fiscal gap narrowed from 72 to 50, which sounds like progress until you realise 50 is still enormous.